# Study 1 — 01 Summary: Reliability and Quality Assessment

Aggregates key findings from all sub-notebooks (01a–01e) into a single summary table.

**Reads from:** `outputs/study1_analysis/tables/*.csv`
**Outputs:** `outputs/study1_analysis/tables/reliability_summary.csv`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
df = build_sentence_df(traces)
hypo_df = df[df['micro_label'] == 'HYPO'].copy()
print(f'Loaded: {df.trace_key.nunique()} traces, {len(df):,} sentences')

In [ ]:
section_header('SUMMARY')

summary = {}

# ── Corpus size ──
summary['total_traces'] = df.trace_key.nunique()
summary['total_sentences'] = len(df)
n_completed = df.groupby('trace_key').completed.first().sum()
summary['completed_traces'] = int(n_completed)
summary['truncated_traces'] = summary['total_traces'] - summary['completed_traces']

# ── Label coverage ──
valid_micro = df['micro_label'].isin(MICRO_LABELS).sum()
summary['micro_label_valid_pct'] = round(valid_micro / len(df) * 100, 2)
valid_macro = df['macro_label'].isin(MACRO_LABELS).sum()
summary['macro_label_valid_pct'] = round(valid_macro / len(df) * 100, 2)

# ── Flag coverage ──
test_df = df[df['micro_label'] == 'TEST']
summary['test_context_valid_pct'] = round(test_df['test_context'].isin(VALID_TEST_CONTEXT).mean() * 100, 1)
summary['specificity_valid_pct'] = round(test_df['specificity'].isin(VALID_SPECIFICITY).mean() * 100, 1)
judge_df = df[df['micro_label'] == 'JUDGE']
summary['judgement_valid_pct'] = round(judge_df['judgement'].isin(VALID_JUDGEMENT).mean() * 100, 1)
summary['confidence_valid_pct'] = round(df['confidence'].isin(VALID_CONFIDENCE).mean() * 100, 1)

# ── Validation κ ──
kappa_df = load_kappa_results()
summary['mean_kappa_micro'] = round(kappa_df['kappa_micro'].mean(), 3)
summary['mean_kappa_macro'] = round(kappa_df['kappa_macro'].mean(), 3)

# Categories passing/failing
cat_cols = [f'kappa_{m}' for m in MICRO_LABELS]
passing = []
failing = []
for m in MICRO_LABELS:
    col = f'kappa_{m}'
    if col in kappa_df.columns:
        mean_k = kappa_df[col].mean()
        if mean_k >= 0.50:
            passing.append(m)
        else:
            failing.append(m)
summary['categories_passing'] = ', '.join(passing)
summary['categories_failing'] = ', '.join(failing) if failing else 'None'

# ── Dependency coverage ──
has_dep = (df['n_dependencies'] > 0).sum()
summary['dependency_coverage_pct'] = round(has_dep / len(df) * 100, 1)
summary['mean_deps_per_sentence'] = round(df['n_dependencies'].mean(), 2)

# ── Hypothesis cycles ──
# Import cycle finder from 01d logic
def find_cycles_simple(trace_df):
    trace_df = trace_df.sort_values('sentence_id').reset_index(drop=True)
    hypo_sids = trace_df.loc[trace_df['micro_label'] == 'HYPO', 'sentence_id'].tolist()
    cycles = []
    for hypo_sid in hypo_sids:
        judges = trace_df[
            (trace_df['micro_label'] == 'JUDGE') &
            (trace_df['sentence_id'] > hypo_sid) &
            (trace_df['depends_on'].apply(lambda deps: hypo_sid in deps))
        ]
        if len(judges) > 0:
            judge_sid = judges.iloc[0]['sentence_id']
            length = len(trace_df[
                (trace_df['sentence_id'] >= hypo_sid) &
                (trace_df['sentence_id'] <= judge_sid)
            ])
            cycles.append(length)
    return cycles

all_cycle_lengths = []
cycles_per_trace = []
for key, grp in df.groupby('trace_key'):
    cl = find_cycles_simple(grp)
    cycles_per_trace.append(len(cl))
    all_cycle_lengths.extend(cl)

summary['mean_cycles_per_trace'] = round(np.mean(cycles_per_trace), 1)
summary['mean_cycle_length'] = round(np.mean(all_cycle_lengths), 1) if all_cycle_lengths else 0
n_hypo = (df['micro_label'] == 'HYPO').sum()
n_hypo_in_cycle = len(all_cycle_lengths)
summary['cycle_completion_rate'] = round(n_hypo_in_cycle / n_hypo * 100, 1) if n_hypo > 0 else 0

# ── Hypothesis recycling ──
status_counts = hypo_df['hypo_status'].value_counts()
for s in ['novel', 'revised', 'repeated']:
    summary[f'pct_hypo_{s}'] = round(status_counts.get(s, 0) / len(hypo_df) * 100, 1)

# Perseverative traces
def max_chain_length(trace_hypos):
    trace_hypos = trace_hypos.sort_values('sentence_id')
    antecedent = {}
    for _, row in trace_hypos.iterrows():
        if pd.notna(row['hypo_antecedent_sid']):
            antecedent[row['sentence_id']] = int(row['hypo_antecedent_sid'])
    all_sids = set(trace_hypos['sentence_id'])
    children = {}
    for sid, ant in antecedent.items():
        if ant in all_sids:
            children.setdefault(ant, []).append(sid)
    roots = [sid for sid in all_sids if sid not in antecedent or antecedent[sid] not in all_sids]
    max_len = 0
    for root in roots:
        length = 1
        current = root
        while current in children:
            current = sorted(children[current])[0]
            length += 1
        max_len = max(max_len, length)
    return max_len

persev_count = 0
for key, grp in hypo_df.groupby('trace_key'):
    if max_chain_length(grp) >= 5:
        persev_count += 1
summary['pct_perseverative_traces'] = round(persev_count / df.trace_key.nunique() * 100, 1)

# Final rule source
completed_df = df[df['completed'] == True]
rule_traces = completed_df[completed_df['micro_label'] == 'RULE']['trace_key'].unique()
novel_rules = 0
total_rules = 0
for key in rule_traces:
    trace = df[df['trace_key'] == key].sort_values('sentence_id')
    rules = trace[trace['micro_label'] == 'RULE']
    if len(rules) == 0:
        continue
    last_rule = rules.iloc[-1]
    rule_deps = last_rule['depends_on']
    judge_sids = trace.loc[
        (trace['sentence_id'].isin(rule_deps)) & (trace['micro_label'] == 'JUDGE'),
        'sentence_id'
    ].tolist()
    if not judge_sids:
        continue
    judge_row = trace[trace['sentence_id'] == judge_sids[0]].iloc[0]
    hypo_sids = trace.loc[
        (trace['sentence_id'].isin(judge_row['depends_on'])) & (trace['micro_label'] == 'HYPO'),
        'sentence_id'
    ].tolist()
    if not hypo_sids:
        continue
    hypo_row = trace[trace['sentence_id'] == hypo_sids[0]].iloc[0]
    total_rules += 1
    if hypo_row.get('hypo_status') == 'novel':
        novel_rules += 1

summary['pct_final_rule_from_novel'] = round(novel_rules / total_rules * 100, 1) if total_rules > 0 else 0

# ── Print summary ──
print(f"Corpus: {summary['total_traces']} traces, {summary['total_sentences']:,} sentences")
print(f"  Completed: {summary['completed_traces']}, Truncated: {summary['truncated_traces']}")
print(f"Label coverage: micro={summary['micro_label_valid_pct']}%, macro={summary['macro_label_valid_pct']}%")
print(f"Flag coverage: test_context={summary['test_context_valid_pct']}%, "
      f"specificity={summary['specificity_valid_pct']}%, "
      f"judgement={summary['judgement_valid_pct']}%, "
      f"confidence={summary['confidence_valid_pct']}%")
print(f"Validation κ: micro={summary['mean_kappa_micro']}, macro={summary['mean_kappa_macro']}")
print(f"  Passing: {summary['categories_passing']}")
print(f"  Failing: {summary['categories_failing']}")
print(f"Dependency coverage: {summary['dependency_coverage_pct']}%")
print(f"Mean deps per sentence: {summary['mean_deps_per_sentence']}")
print(f"Mean cycles per trace: {summary['mean_cycles_per_trace']} (length: {summary['mean_cycle_length']})")
print(f"Cycle completion rate: {summary['cycle_completion_rate']}%")
print(f"Hypothesis status: novel={summary['pct_hypo_novel']}%, "
      f"revised={summary['pct_hypo_revised']}%, repeated={summary['pct_hypo_repeated']}%")
print(f"Perseverative traces: {persev_count}/{summary['total_traces']} ({summary['pct_perseverative_traces']}%)")
print(f"Final rules from novel HYPO: {summary['pct_final_rule_from_novel']}%")

# Save
summary_df = pd.DataFrame([summary])
save_table(summary_df, 'reliability_summary.csv')
display(summary_df.T)